- **Name:** 20.5_streaming_aggregations
- **Author:** Shamas Imran
- **Desciption:** Performing aggregations in structured streaming
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Applied groupBy aggregations on streams  
                                              Used tumbling and sliding windows  
                                              Demonstrated append vs update modes  
-->

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import col, window, session_window

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 13, Finished, Available, Finished)

In [12]:
# ------------------------------------------------------------
# 1) Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("Aggregations_and_WindowedAggregations")
        .getOrCreate()
)

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 14, Finished, Available, Finished)

In [13]:
rootPath = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/"
masterPath = rootPath + "spark-streaming/"
inputPath       = masterPath + "csv_input"
checkpointPath  = masterPath + "checkpoints/aggregations"
outputPath      = masterPath + "csv_output"

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 15, Finished, Available, Finished)

In [14]:
# Check if folder exists before deleting
if mssparkutils.fs.exists(masterPath):
    mssparkutils.fs.rm(masterPath, True)  # True = recursive delete

# Create directories
import os

os.makedirs(masterPath, exist_ok=True)
os.makedirs(inputPath, exist_ok=True)
os.makedirs(checkpointPath, exist_ok=True)
os.makedirs(outputPath, exist_ok=True)

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 16, Finished, Available, Finished)

In [15]:
import pandas as pd
import os

lakehouse_folder = inputPath  # your Lakehouse input path

data = [
    {"id": 1,  "name": "Ahsan", "score": 85, "event_time": "2025-08-18 10:00:00"},
    {"id": 2,  "name": "Sana",  "score": 92, "event_time": "2025-08-18 10:01:00"},
    {"id": 3,  "name": "Ali",   "score": 78, "event_time": "2025-08-18 10:02:00"},
    {"id": 4,  "name": "Ahsan", "score": 88, "event_time": "2025-08-18 10:03:00"},
    {"id": 5,  "name": "Sana",  "score": 95, "event_time": "2025-08-18 10:04:00"},
    {"id": 6,  "name": "Ali",   "score": 82, "event_time": "2025-08-18 10:06:00"},
    {"id": 7,  "name": "Ahsan", "score": 90, "event_time": "2025-08-18 10:07:00"},
    {"id": 8,  "name": "Sana",  "score": 87, "event_time": "2025-08-18 10:09:00"},
    {"id": 9,  "name": "Ali",   "score": 91, "event_time": "2025-08-18 10:10:00"},
    {"id": 10, "name": "Ahsan", "score": 80, "event_time": "2025-08-18 09:50:00"},  # Late, within 10-min watermark
    {"id": 11, "name": "Sana",  "score": 85, "event_time": "2025-08-18 09:40:00"},  # Too late, will be dropped by watermark
    {"id": 12, "name": "Ali",   "score": 88, "event_time": "2025-08-18 10:12:00"},
    {"id": 13, "name": "Ahsan", "score": 92, "event_time": "2025-08-18 10:14:00"},
    {"id": 14, "name": "Sana",  "score": 90, "event_time": "2025-08-18 10:15:00"},
    {"id": 15, "name": "Ali",   "score": 83, "event_time": "2025-08-18 10:18:00"},
]

valid_df = pd.DataFrame(data)
valid_path = os.path.join(lakehouse_folder, "Valid_data.csv")
valid_df.to_csv(valid_path, index=False, header=True)

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 17, Finished, Available, Finished)

In [16]:
df = spark.read.format("csv").option("header","true").load(f"{inputPath}/Valid_data.csv").sort("event_time")
df.show()

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 18, Finished, Available, Finished)

+---+-----+-----+-------------------+
| id| name|score|         event_time|
+---+-----+-----+-------------------+
| 11| Sana|   85|2025-08-18 09:40:00|
| 10|Ahsan|   80|2025-08-18 09:50:00|
|  1|Ahsan|   85|2025-08-18 10:00:00|
|  2| Sana|   92|2025-08-18 10:01:00|
|  3|  Ali|   78|2025-08-18 10:02:00|
|  4|Ahsan|   88|2025-08-18 10:03:00|
|  5| Sana|   95|2025-08-18 10:04:00|
|  6|  Ali|   82|2025-08-18 10:06:00|
|  7|Ahsan|   90|2025-08-18 10:07:00|
|  8| Sana|   87|2025-08-18 10:09:00|
|  9|  Ali|   91|2025-08-18 10:10:00|
| 12|  Ali|   88|2025-08-18 10:12:00|
| 13|Ahsan|   92|2025-08-18 10:14:00|
| 14| Sana|   90|2025-08-18 10:15:00|
| 15|  Ali|   83|2025-08-18 10:18:00|
+---+-----+-----+-------------------+



In [17]:
# for s in spark.streams.active:
#     s.stop()

# spark.sql("DROP TABLE IF EXISTS test_Lakehouse.agg_no_watermark_query")
# spark.sql("DROP TABLE IF EXISTS test_Lakehouse.agg_with_watermark_query")

# mssparkutils.fs.rm("Tables/agg_no_watermark_query", recurse=True)
# mssparkutils.fs.rm("Tables/agg_with_watermark_query", recurse=True)

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 19, Finished, Available, Finished)

In [18]:
# ------------------------------------------------------------
# 3) Define Schema
# ------------------------------------------------------------
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("event_time", TimestampType(), True)
])

# ------------------------------------------------------------
# 4) Create Streaming DataFrame
# ------------------------------------------------------------
df_stream = (
    spark.readStream
         .option("header", "true")
         .schema(schema)
         .csv(inputPath)
)

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 20, Finished, Available, Finished)

In [20]:
# ------------------------------------------------------------
# 5) Aggregation WITHOUT Watermark
# ------------------------------------------------------------
# Stateful aggregation without watermark (state can grow indefinitely)
agg_no_watermark = df_stream.groupBy("name").count()

agg_no_watermark_query = (
    agg_no_watermark.writeStream
                .format("delta")                       # ✅ Delta table sink
                .option("checkpointLocation", checkpointPath + "/no_watermark")
                .outputMode("complete")                # required for aggregations
                .trigger(processingTime="10 seconds")  # micro-batch interval
                .toTable("agg_no_watermark_query")
)

agg_no_watermark_query.awaitTermination()

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 22, Finished, Cancelled, Cancelled)

In [21]:
df = spark.sql("SELECT * FROM test_Lakehouse.agg_no_watermark_query LIMIT 1000")
df.show()

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, 23, Finished, Available, Finished)

+-----+-----+
| name|count|
+-----+-----+
|Ahsan|    5|
| Sana|    5|
|  Ali|    5|
+-----+-----+



In [ ]:
"""
id,name,score,event_time
1,Ahsan,85,2025-08-18 10:00:00
2,Sana,92,2025-08-18 10:01:00
3,Ali,78,2025-08-18 10:02:00
4,Ahsan,88,2025-08-18 10:03:00
5,Sana,95,2025-08-18 10:04:00
6,Ali,82,2025-08-18 10:06:00
7,Ahsan,90,2025-08-18 10:07:00
8,Sana,87,2025-08-18 10:09:00
9,Ali,91,2025-08-18 10:10:00
10,Ahsan,80,2025-08-18 09:50:00   # Late, within 10-min watermark
11,Sana,85,2025-08-18 09:40:00    # Too late, will be dropped by watermark
12,Ali,88,2025-08-18 10:12:00
13,Ahsan,92,2025-08-18 10:14:00
14,Sana,90,2025-08-18 10:15:00
15,Ali,83,2025-08-18 10:18:00
"""

StatementMeta(, a902bb64-cc02-4054-be5f-c924e702314a, -1, Cancelled, , Cancelled)